# 09 — Lane Detection with Canny + Hough Transform

**Section:** Automated Driving · **Mirrors MATLAB:** *Lane Following Control with Sensor Fusion and Lane Detection*

Classical lane detection pipeline:

1. Convert to grayscale.
2. Apply **Canny** edge detection.
3. Mask a **region-of-interest** trapezoid (road ahead).
4. Run the **probabilistic Hough transform** to extract line segments.

We synthesize a road scene so the notebook needs no external images.

## Intuition — what's actually going on?

A self-driving car needs to know where the lane lines are. The very oldest and still very useful technique is purely "classical" image processing — no neural networks required:

1. **Edges**: lane lines are sharp boundaries between road and paint. Compute the *gradient* of brightness at every pixel; high gradient = edge. That's what **Canny** does (with some clever extras: thin the edges to 1-pixel-wide, then chain them).
2. **Region of interest**: lane lines only appear in a trapezoid in front of the car. Mask out everything else (sky, oncoming lane, hood). Crucial: most of the "edges" we found are not lanes — we just throw them away.
3. **Lines**: edge pixels alone aren't lines. The **Hough transform** is a vote-counting scheme — every edge pixel "votes" for all possible lines that pass through it, and lines with many votes win.

The result is a list of line segments that match the lanes. From there it's a short hop to "where's the lane center and how do I steer to stay in it" (see notebook 06 for the steering controller).

### Analytical underpinning + compatibility

**Canny edges.** For each pixel compute gradient magnitude

$$\|
abla I\| = \sqrt{I_x^2 + I_y^2},\qquad I_x = \partial I / \partial x,\ I_y = \partial I / \partial y$$

via Sobel kernels, suppress non-maxima along the gradient direction, then hysteresis-threshold (`low`, `high`): keep pixels above `high`, and pixels above `low` that are connected to a `high` pixel.

**Hough line transform.** A line $ho = x\cos	heta + y\sin	heta$ is parameterized by $(ho, 	heta)$. For each edge pixel $(x_i, y_i)$, vote in $(ho, 	heta)$-space along every curve $ho = x_i\cos	heta + y_i\sin	heta$. Peaks in the accumulator correspond to lines that pass through many edge pixels.

The *probabilistic* Hough variant samples a random subset of edge pixels (much faster) and returns line *segments* with explicit endpoints rather than the full $(ho, 	heta)$ line.

### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| Sobel-based gradient + hysteresis | `edges = cv2.Canny(gray, 60, 160)` (low=60, high=160) |
| ROI trapezoid mask | `cv2.fillPoly(mask, roi, 255); edges_roi = cv2.bitwise_and(edges, mask)` |
| Probabilistic Hough line segments | `cv2.HoughLinesP(edges_roi, 1, np.pi/180, 40, minLineLength=35, maxLineGap=25)` (resolution: 1 px, 1°; vote threshold 40) |


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

np.random.seed(1)

H, W = 360, 640
img = np.full((H, W, 3), (60, 60, 60), dtype=np.uint8)
# Road surface
pts_road = np.array([[100, H], [W - 100, H], [W // 2 + 70, H - 220], [W // 2 - 70, H - 220]])
cv2.fillPoly(img, [pts_road], (95, 95, 95))
# Center dashed line
for y in range(H - 200, H, 35):
    cv2.line(img, (W // 2, y), (W // 2, y + 18), (240, 220, 60), 3)
# Solid white edges
cv2.line(img, (130, H - 5), (W // 2 - 60, H - 220), (240, 240, 240), 5)
cv2.line(img, (W - 130, H - 5), (W // 2 + 60, H - 220), (240, 240, 240), 5)
img = cv2.GaussianBlur(img, (3, 3), 0)
noise = np.random.randint(-12, 12, img.shape, dtype=np.int16)
img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)


In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 60, 160)

mask = np.zeros_like(edges)
roi = np.array([[(60, H), (W - 60, H), (W // 2 + 60, H - 220), (W // 2 - 60, H - 220)]])
cv2.fillPoly(mask, roi, 255)
edges_roi = cv2.bitwise_and(edges, mask)

lines = cv2.HoughLinesP(edges_roi, 1, np.pi / 180, threshold=40,
                         minLineLength=35, maxLineGap=25)

out = img.copy()
if lines is not None:
    for l in lines:
        x1, y1, x2, y2 = l[0]
        cv2.line(out, (x1, y1), (x2, y2), (0, 255, 0), 3)
print(f"Detected {0 if lines is None else len(lines)} line segments")


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
axs[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axs[0].set_title('Synthetic input')
axs[1].imshow(edges_roi, cmap='gray');               axs[1].set_title('Canny edges (ROI)')
axs[2].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); axs[2].set_title('Hough lanes (green)')
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()


## References & rigor notes

**Canny optimality** (Canny, 1986). The Canny detector is optimal in a specific sense: for an *idealized 1-D step edge with additive Gaussian noise*, it satisfies (i) good detection (low miss + false-alarm rate), (ii) good localization (detected edges close to true edges), and (iii) one response per edge. These three criteria are encoded as variational objectives whose solution is approximately the derivative of a Gaussian; Canny shows the explicit closed form. For real images with shadows, texture, or wide gradient transitions, the optimality is only approximate.

**Hough transform.** Each edge pixel votes for *all* lines passing through it in $(\rho, \theta)$ space (the line $\rho = x\cos\theta + y\sin\theta$). Lines with votes above threshold are returned. Complexity $O(N \cdot K)$ where $N$ = edge pixels and $K$ = number of $\theta$ bins. With $K = 180$ bins and $N = O(\text{image pixels})$ edges, classical Hough is $\sim 10^8$ ops for HD images — only real-time via the **probabilistic** variant (Matas-Galambos-Kittler 2000) that samples a random subset of edge pixels.

**Threshold tuning** (council fix, Borrelli): the vote threshold `40` is hand-tuned to this $360 \times 640$ synthetic image. Production AV stacks scale threshold with image size: $\text{threshold} \approx k \cdot \min(H, W)$. The static trapezoidal ROI also fails on hills; production uses IMU-pitch-adaptive ROI.

**Limitations.** Canny + Hough handles only straight-line lanes; curved highways require polynomial fitting on the warped (bird's-eye) image, or deep-learning segmentation networks.

**References.**
- Canny, J. (1986). *A computational approach to edge detection*. IEEE Trans. PAMI, 8(6), 679-698.
- Duda, R. O., & Hart, P. E. (1972). *Use of the Hough transformation to detect lines and curves in pictures*. CACM, 15(1), 11-15.
- Matas, J., Galambos, C., & Kittler, J. (2000). *Robust detection of lines using the progressive probabilistic Hough transform*. CVIU, 78(1).
